In [2]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
Dir_github = '/work/desai-lab/xuanyang/Project/ABC/github'
Dir_dataset = '/work/desai-lab/ABC_dataset'
Dir_preprocess = os.path.join(Dir_dataset,'derivatives','fmriprep')


In [3]:
def extract_compcor(path_confounds,tissue,N_comps):
    df_confounds = pd.read_csv(path_confounds,sep='\t')
    path_confounds_json = path_confounds.replace('.tsv','.json')
    f = open (path_confounds_json, "r")
    data = json.loads(f.read())
    f.close()
    
    
    List_compcor_all = [x for x in df_confounds.columns if 'comp_cor' in x and 't_comp_cor' not in x]
    List_aCompCor = []
    df = []
    List_cosine = [x for x in df_confounds.columns if 'cosine' in x]
    if len(List_cosine)>0 and len(List_compcor_all)>0:


        for x in List_compcor_all:
            if data[x]['Mask'] == tissue and data[x]['Method']=='aCompCor':
                List_aCompCor.append(x)
                df.append(pd.DataFrame({'c':x,'Mask':data[x]['Mask'],'Method':data[x]['Method'],
                                      'Retained':data[x]['Retained'],'SingularValue':data[x]['SingularValue'],
                                      'VarExp':data[x]['VarianceExplained']},index=[0]))
        df = pd.concat(df,ignore_index=True)
        df = df.loc[df.Retained==1].sort_values(['SingularValue'],ascending=False)

        if len(List_aCompCor)>=N_comps:
            List_aCompCor = List_aCompCor[:N_comps]


        List_cosine = [x for x in df_confounds.columns if 'cosine' in x]
        
    else:
        print('Failed to extract cosine or aCompCor. Check {}'.format(path_confounds))
#         print('Failed to extract cosine or aCompCor.\n')
        
    return(df,List_cosine, List_aCompCor)

In [4]:
df_master = pd.read_csv(os.path.join(Dir_github,'derivatives','test_Fam','df_master_Fam120.csv'))
df_master

,subid,visit,Fam_fmri,Fam_eprime,age,sex,trans_x,trans_y,trans_z,rot_x,rot_y,rot_z,motion_max
0,1005,1.0,1.0,1.0,65.0,2.0,0.041617,0.267869,0.463099,0.007059,0.003916,0.003123,0.463099
1,1007,1.0,1.0,1.0,69.0,2.0,0.383059,0.671302,0.386057,0.009304,0.001248,0.003724,0.671302
2,1008,1.0,1.0,1.0,77.0,2.0,0.075486,0.203145,0.118612,0.005849,0.000214,0.001656,0.203145
3,1009,1.0,1.0,1.0,75.0,2.0,0.000065,0.349682,0.215196,0.003958,0.000177,0.000196,0.349682
4,1018,1.0,1.0,1.0,60.0,2.0,0.026452,0.412780,0.433625,0.003764,0.005943,0.000393,0.433625
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,1302,1.0,1.0,1.0,23.0,1.0,0.565818,0.318678,0.295166,0.004322,0.001432,0.012690,0.565818
116,1305,1.0,1.0,1.0,61.0,2.0,0.061026,0.375817,0.324899,0.005508,0.000556,0.000923,0.375817
117,1306,1.0,1.0,1.0,28.0,2.0,0.485715,0.411004,0.707648,0.018751,0.001389,0.016267,0.707648
118,1311,1.0,1.0,1.0,20.0,2.0,0.174499,0.285192,0.111143,0.003649,0.008804,0.005942,0.285192


# Pipeline

In [20]:
nWM = 3
nCSF = 3
nPolort = 6
nMotion = 6
if nMotion == 6:
    List_motion = ['trans_x', 'trans_y', 'trans_z','rot_x', 'rot_y', 'rot_z']
censor = 1

pipeline = f"test_Fam_mot{nMotion}csf{nCSF}wm{nWM}plt{nPolort}cen{censor}"
Dir_pipeline = os.path.join(Dir_github,'derivatives',pipeline)
if not os.path.exists(Dir_pipeline):
    os.makedirs(Dir_pipeline)
    os.makedirs(os.path.join(Dir_pipeline,'scripts'))
pipeline

'test_Fam_mot6csf3wm3plt6cen1'

In [21]:
for i, row in df_master.iterrows():
    

    subid = int(row['subid'])
    visit = int(row['visit'])
    Dir_subject = os.path.join(Dir_preprocess,f"sub-{subid}",f"ses-visit0{visit}")

    # T1 mask
    mask = os.path.join(Dir_subject,'anat',f'sub-{subid}_ses-visit0{visit}_space-MNI152NLin2009cAsym_res-2_desc-brain_mask.nii.gz')
    if not os.path.exists(mask):
        mask = os.path.join(Dir_preprocess,f"sub-{subid}",'anat',f'sub-{subid}_space-MNI152NLin2009cAsym_res-2_desc-brain_mask.nii.gz')
        if not os.path.exists(mask):
            print(subid)
    # fmri image

    img = os.path.join(Dir_subject,'func',f"sub-{subid}_ses-visit0{visit}_task-Fam_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz")

    # confounding variables
    path_confounds = os.path.join(Dir_subject,'func',f"sub-{subid}_ses-visit0{visit}_task-Fam_desc-confounds_timeseries.tsv")
    df_confounds = pd.read_csv(path_confounds,sep='\t')
    list_outlier = [outlier for outlier in df_confounds.columns if 'motion_outlier' in outlier]
    df_confounds['motion_outlier'] = 1-df_confounds[list_outlier].sum(axis=1)

    df_tmp,List_cosine,List_aCompCor_CSF = extract_compcor(path_confounds,'CSF',nCSF)
    df_tmp,List_cosine,List_aCompCor_WM = extract_compcor(path_confounds,'WM',nWM)

    df_confounds_model = df_confounds[List_motion+List_cosine+List_aCompCor_CSF+List_aCompCor_WM]


    Dir_output = os.path.join(Dir_pipeline,'results',f'sub-{subid}')
    if not os.path.exists(Dir_output):
        os.makedirs(Dir_output)


    path_motion_outlier = os.path.join(Dir_output,f'sub-{subid}_ses-visit0{visit}_task-Fam_motionsCensor.1D')
    df_confounds['motion_outlier'].to_csv(path_motion_outlier,index=False,header=False,sep=' ')

    path_confounds = os.path.join(Dir_output,f'sub-{subid}_ses-visit0{visit}_task-Fam_confounds_{pipeline}.1D')
    if not os.path.exists(path_confounds):
        df_confounds_model.to_csv(path_confounds,index=False,header=False,sep=' ')


    # script to run
    scriptFile = os.path.join(Dir_output,f'3dDeconvolve_{subid}.sh')

    with open(scriptFile, "w") as f:
        f.write("#!/bin/tcsh\n")
        f.write(f"/work/apps/AFNI/linux_openmp_64/afni_proc.py -subj_id sub-{subid} \\\n")
        f.write("-scr_overwrite -blocks mask regress \\\n")
        f.write(f"-dsets {img} \\\n")
        f.write(f"-regress_motion_file {path_confounds} \\\n")
        f.write(f"-regress_censor_extern {path_motion_outlier} \\\n")
        f.write("-regress_opts_3dD -bout -jobs 6 \\\n")
        f.write(f"-regress_polort {nPolort} \\\n")        
        f.write("-regress_make_ideal_sum sum_ideal.1D \\\n")
        f.write("-regress_run_clustsim no \\\n")
        f.write("\n")

    masterScript = os.path.join(Dir_pipeline,'scripts',f'run_3dDeconvolve_{subid}.sh')

    with open(masterScript, "w") as f:
        f.write("#!/bin/tcsh\n")
        f.write(f"cd {os.path.join(Dir_output)} \n")
        f.write(f"tcsh {scriptFile} \n")
        f.write(f"tcsh proc.sub-{subid} \n")
        f.write(f"rm {Dir_output}/sub-{subid}.results/pb00.* \n")
    print(subid,'generated correctly.')

    
    

1005 generated correctly.
1007 generated correctly.
1008 generated correctly.
1009 generated correctly.
1018 generated correctly.
1019 generated correctly.
1020 generated correctly.
1021 generated correctly.
1024 generated correctly.
1025 generated correctly.
1030 generated correctly.
1032 generated correctly.
1037 generated correctly.
1041 generated correctly.
1042 generated correctly.
1047 generated correctly.
1048 generated correctly.
1049 generated correctly.
1051 generated correctly.
1052 generated correctly.
1053 generated correctly.
1055 generated correctly.
1056 generated correctly.
1060 generated correctly.
1062 generated correctly.
1065 generated correctly.
1069 generated correctly.
1070 generated correctly.
1072 generated correctly.
1073 generated correctly.
1076 generated correctly.
1078 generated correctly.
1079 generated correctly.
1081 generated correctly.
1083 generated correctly.
1085 generated correctly.
1086 generated correctly.
1087 generated correctly.
1091 generat

In [22]:
Dir_logs = os.path.join(Dir_pipeline,'logs')
if not os.path.exists(Dir_logs):
    os.makedirs(Dir_logs)

jobname = pipeline
ncores = 6
partition = 'desailab-48core'
script_slurm = os.path.join(Dir_pipeline,'scripts','Batch_slurm_3dDeconvolve.sh')
array = '1001-1323'
#array = '001-013'
with open(script_slurm, "w") as f:
    f.write("#!/bin/sh\n")
    f.write("### name your job \n")
    f.write(f"#SBATCH --job-name={jobname} \n")
    f.write("### specify number of cores to use \n")
    f.write(f"#SBATCH -n {ncores} \n")
    f.write("#SBATCH -N 1 \n".format(ncores))
    f.write("### which partition do you want to use? \n")
    f.write(f"#SBATCH -p {partition} \n")
    f.write("### %a specifies that the output file for each subject will be named according to their array ID \n")
    f.write("# Where to output log files? \n")
    f.write(f"#SBATCH --output='{Dir_logs}/{pipeline}-%A_%a.log' \n")
    f.write(f"#SBATCH --error='{Dir_logs}/{pipeline}-%A_%a.err' \n")
    f.write("### list the subjects that you want to submit in this array \n")
    f.write(f"#SBATCH --array={array} \n")
    f.write("module load afni \n")
    f.write("module load python3/anaconda/2021.11 \n")
    f.write('echo "Slurm job ID: " $SLURM_JOB_ID \n')
    f.write('echo "Slurm array task ID: " $SLURM_ARRAY_TASK_ID \n')

    f.write('printf -v subj "%03d" $SLURM_ARRAY_TASK_ID \n')
    f.write('echo "Running 3dTproject on sub-$subj" \n')
    f.write(f"cd {os.path.join(Dir_pipeline,'scripts')} \n")
    f.write('tcsh run_3dDeconvolve_$subj.sh \n')
    f.write('echo "Finished 3dDeconvolve sub-$subj" \n')
    f.write('date \n')
    